# Stochastic grokking на $S_n$

Быстрый экспериментальный протокол на основе stochastic-настроек Power et al.: bias-free MLP, AdamW, batch 512, warmup 10. Для $S_5$ stochastic grokking опубликован; перенос на $S_6/S_7$ является экспериментом, поэтому сначала запускаем $S_6$.

## Загрузка скрипта с обязательной проверкой версии

In [ ]:
from pathlib import Path
import shutil, sys

EXPECTED_BUILD = "sn-stochastic-power-v1.3-wandb-2026-08-03"
candidates = list(Path("/kaggle/input").rglob("train_sn_stochastic.py"))
if not candidates:
    raise FileNotFoundError("Добавьте новый train_sn_stochastic.py как Kaggle Input")
source = candidates[0]
local = Path("/kaggle/working/train_sn_stochastic.py")
shutil.copy2(source, local)
sys.path.insert(0, "/kaggle/working")
from train_sn_stochastic import BUILD_ID, Config, run
print("Script:", source)
print("Build:", BUILD_ID)
assert BUILD_ID == EXPECTED_BUILD

## Финальная настройка $S_6$: чуть более выраженное плато

Рабочий режим `f=0.30, batch=512, wd=0.05` дал настоящий threshold-gap 39 600 шагов. Чтобы разнести кривые ещё немного, сохраняем split, seed, batch, lr и архитектуру и делаем минимальный шаг **`weight_decay: 0.05 → 0.045`**. Это не новый поиск режима, а осторожная локальная интерполяция около уже успешной точки.

Ожидаемый threshold-gap — примерно 45–50k шагов. Новый каталог `..._wd0045`, бюджет 500k, после grokking сохраняются ещё 15k шагов для EDM.


In [ ]:
CONFIG = Config(
    output_root="/kaggle/working/sn_stochastic_grokking",
    protocol_name="power_adamw_stochastic_v1_b512_f30_wd0045",
    n_values=(6,),
    seeds=(42,),
    train_fraction=0.30,
    batch_size_by_n={5: 512, 6: 512, 7: 512},
    max_steps_by_n={5: 150_000, 6: 500_000, 7: 600_000},
    learning_rate=1e-3,
    weight_decay=0.045,
    betas=(0.9, 0.98),
    warmup_steps=10,
    log_every=20,
    diagnostic_every=1_000,
    checkpoint_every=2_000,
    tensorboard=True,
    tensorboard_flush_secs=30,
    required_gap_steps=45_000,
    post_grok_steps=15_000,
    use_amp=False,
    resume_search_roots=("/kaggle/input",),
)
CONFIG

## TensorBoard

События сохраняются отдельно для каждого запуска в `tensorboard/`. Запустите следующую ячейку **до обучения**, тогда графики можно обновлять прямо во время эксперимента. CSV-логи при этом остаются основным архивным форматом.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/sn_stochastic_grokking

In [ ]:
RUN_DIRS = run(CONFIG)
RUN_DIRS

## Просмотр логов

In [ ]:
import json, pandas as pd
import matplotlib.pyplot as plt
for run_dir in RUN_DIRS:
    frame = pd.read_csv(run_dir / "training_log.csv")
    display(frame.tail())
    print(json.dumps(json.loads((run_dir / "COMPLETED.json").read_text()), indent=2))
    fig, axes = plt.subplots(1, 2, figsize=(13,4))
    axes[0].plot(frame.step, frame.train_loss, label="train")
    axes[0].plot(frame.step, frame.val_loss, label="val")
    axes[0].set_yscale("log"); axes[0].legend(); axes[0].grid(alpha=.25)
    axes[1].plot(frame.step, frame.train_acc, label="train")
    axes[1].plot(frame.step, frame.val_acc, label="val")
    axes[1].legend(); axes[1].grid(alpha=.25)
    plt.show()

## Критерий выбора финального режима

- baseline `wd=0.10`: gap 13 320;
- хороший `wd=0.05`: gap 39 600;
- кандидат `wd=0.045`: целевой gap ≥45 000 и визуальное плато между переходами.

Если `wd=0.045` не обобщит за 500k, не уменьшайте decay дальше: используйте уже успешный `wd=0.05`. Если grokking произойдёт, `wd=0.045` становится финальным режимом.


In [ ]:
# Fallback только если wd=0.045 не обобщил за 500k:
# вернуться к уже подтверждённому wd=0.05, а не продолжать уменьшение decay.
# CONFIG.protocol_name = "power_adamw_stochastic_v1_b512_f30_wd005_repeat"
# CONFIG.weight_decay = 0.05
# CONFIG.required_gap_steps = 25_000
# CONFIG.force_restart = True
# RUN_DIRS = run(CONFIG)


## $S_5$ и $S_7$

Для sanity-check можно поставить `n_values=(5,)`. К $S_7$ переходите только после успешного $S_6$: гарантированного опубликованного режима для него нет.